# M4 unpaired comparison (Robotic Task Breaker vs bare on drawer chain)

Colab launcher for the M4 unpaired protocol on `open_close_drawer`: install deps, start VLA servers, run `X` episodes per arm, interim Fisher stats, then continue to 50/arm when interim is not decisive.

Orchestration lives in `validation/run_robotic_task_breaker_arm.py`, `validation/run_bare_policy.py`, and `validation/m4_unpaired_stats.py`.


## 1 — Clone Robotic Task Breaker and mount Drive

Colab runtime: **Linux + NVIDIA GPU**. Set `GITHUB_TOKEN` (repo PAT) and `OPENAI_API_KEY` in Colab **Secrets** (optional `HF_TOKEN` for gated checkpoint download).

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive, userdata

github_token = userdata.get('GITHUB_TOKEN')
openai_api_key = userdata.get('OPENAI_API_KEY')

if os.path.exists('/content/robotic-task-breaker'):
    os.chdir('/content/robotic-task-breaker')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/robotic-task-breaker.git"
    !git clone $CLONE_URL "/content/robotic-task-breaker"
    os.chdir('/content/robotic-task-breaker')

drive.mount('/content/drive')

RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ARTIFACT_DIR = Path(f'/content/drive/MyDrive/Colab/robotic_task_breaker_debug_drawer_chain/{RUN_UTC}')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['ARTIFACT_DIR'] = str(ARTIFACT_DIR)
os.environ['RUN_UTC'] = RUN_UTC
print('ARTIFACT_DIR=', ARTIFACT_DIR)
print('RUN_UTC=', RUN_UTC)

## 2 — Headless rendering + optional Hugging Face auth

In [ ]:
import os

os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
except Exception:
    pass

## 3 — Isaac-GR00T RoboCasa env + Robotic Task Breaker installs

Minimal n1d6 RoboCasa stack (same `setup_RoboCasa.sh` patch as the M1 gate notebook). Robotic Task Breaker runs in the RoboCasa sim venv; the GR00T HTTP adapter uses the Isaac-GR00T project venv.

In [ ]:
%%bash
set -euo pipefail
SETUP_LOG="/tmp/robotic_task_breaker_m2_setup_${RUN_UTC:-manual}.log"
exec > >(tee "${SETUP_LOG}") 2>&1
trap 'cp -f "${SETUP_LOG}" "${ARTIFACT_DIR}/setup.log" 2>/dev/null || true' EXIT

cd /content/robotic-task-breaker
GROOT_ROOT=/content/Isaac-GR00T
GROOT_BRANCH=n1d6
MODEL_PATH=nvidia/GR00T-N1.6-3B
EMBODIMENT_TAG=ROBOCASA_PANDA_OMRON

for attempt in 1 2 3 4; do
  if apt-get update -qq; then
    break
  fi
  if [ "${attempt}" -eq 4 ]; then
    echo "apt-get update failed after ${attempt} attempts" >&2
    exit 1
  fi
  sleep $((attempt * 5))
done
for attempt in 1 2 3 4; do
  if DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
    git git-lfs ffmpeg libaio-dev libegl1-mesa-dev libglu1-mesa \
    python3-venv python3-dev; then
    break
  fi
  if [ "${attempt}" -eq 4 ]; then
    echo "apt-get install failed after ${attempt} attempts" >&2
    exit 1
  fi
  sleep $((attempt * 5))
done
git lfs install

if [ ! -d "${GROOT_ROOT}/.git" ]; then
  git clone --branch "${GROOT_BRANCH}" --recurse-submodules \
    https://github.com/NVIDIA/Isaac-GR00T.git "${GROOT_ROOT}"
else
  git -C "${GROOT_ROOT}" fetch origin "${GROOT_BRANCH}"
  git -C "${GROOT_ROOT}" checkout "${GROOT_BRANCH}"
  git -C "${GROOT_ROOT}" submodule update --init --recursive
fi

cd "${GROOT_ROOT}"
curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"

python3 - <<'PY'
from pathlib import Path
script = Path('gr00t/eval/sim/robocasa/setup_RoboCasa.sh')
text = script.read_text(encoding='utf-8')
needle = 'uv pip install '
patched = 'uv pip install --python "$UV_ENV/.venv/bin/python" '
if patched not in text:
    script.write_text(text.replace(needle, patched), encoding='utf-8')
PY

export UV_HTTP_TIMEOUT=120
for attempt in 1 2 3 4; do
  if uv sync; then
    break
  fi
  if [ "${attempt}" -eq 4 ]; then
    echo "uv sync failed after ${attempt} attempts" >&2
    exit 1
  fi
  sleep $((attempt * 15))
done
source .venv/bin/activate
bash gr00t/eval/sim/robocasa/setup_RoboCasa.sh

ROBO_PY="${GROOT_ROOT}/gr00t/eval/sim/robocasa/robocasa_uv/.venv/bin/python"
GROOT_PY="${GROOT_ROOT}/.venv/bin/python"

cd /content/robotic-task-breaker
"${ROBO_PY}" -m ensurepip --upgrade || true
"${ROBO_PY}" -m pip install -U pip
"${ROBO_PY}" -m pip install -r deps/requirements.txt
"${ROBO_PY}" -m pip install -e .

uv pip install --python "${GROOT_PY}" -r deps/requirements.txt
uv pip install --python "${GROOT_PY}" -e .

echo "GROOT_ROOT=${GROOT_ROOT}" > /content/robotic-task-breaker/.colab_m2_env
echo "ROBO_PY=${ROBO_PY}" >> /content/robotic-task-breaker/.colab_m2_env
echo "GROOT_PY=${GROOT_PY}" >> /content/robotic-task-breaker/.colab_m2_env
echo "MODEL_PATH=${MODEL_PATH}" >> /content/robotic-task-breaker/.colab_m2_env
echo "EMBODIMENT_TAG=${EMBODIMENT_TAG}" >> /content/robotic-task-breaker/.colab_m2_env

"${ROBO_PY}" - <<'PY'
import gymnasium as gym
import robocasa  # noqa: F401
from robocasa.utils.gym_utils import GrootRoboCasaEnv  # noqa: F401
import robosuite  # noqa: F401
from envs.robocasa_env import OPEN_DRAWER_GYM_ENV_ID

try:
    gym.spec(OPEN_DRAWER_GYM_ENV_ID)
except Exception as exc:
    namespaces = sorted(
        {
            spec.namespace
            for spec in gym.envs.registry.values()
            if getattr(spec, "namespace", None)
        }
    )
    print(f"missing gym spec: {OPEN_DRAWER_GYM_ENV_ID}")
    print(f"registered namespaces (sample): {namespaces[:20]}")
    raise RuntimeError("RoboCasa PandaOmron gym namespace is not registered") from exc

print(f"RoboCasa + Robotic Task Breaker import OK; gym spec ready: {OPEN_DRAWER_GYM_ENV_ID}")
PY

## 4 — Configure `.env` (RoboCasa + GR00T)


In [ ]:
import os
from pathlib import Path

X = 20  # interim episodes per arm
FINAL_N = 50  # final episodes per arm if interim is not decisive
os.environ['M4_X'] = str(X)
os.environ['M4_FINAL_N'] = str(FINAL_N)

Path('.env').write_text(
    f'OPENAI_API_KEY={openai_api_key}\n'
    'VLA_POLICY=groot\n'
    'LOG_LEVEL=DEBUG\n'
    'LLM_MODEL=gpt-4.1-mini\n'
    'SAVE_VIDEO=1\n'
    f'VIDEO_PATH={ARTIFACT_DIR / "run.mp4"}\n',
    encoding='utf-8',
)
print('.env written (SIM_ENV=robocasa is default in src/config.py)')
print('X=', X, 'FINAL_N=', FINAL_N)


## 5 — Start upstream GR00T (:5555) and Robotic Task Breaker adapter (:8765)

In [ ]:
%%bash
set -euo pipefail
source /content/robotic-task-breaker/.colab_m2_env
cd /content/robotic-task-breaker

UPSTREAM_LOG="${ARTIFACT_DIR}/upstream_gr00t_server.log"
ADAPTER_LOG="${ARTIFACT_DIR}/robotic_task_breaker_groot_adapter.log"

pkill -f run_gr00t_server.py || true
pkill -f services/groot_service.py || true
pkill -f services/run_vla_server.py || true
sleep 2

export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export PYTHONUNBUFFERED=1

cd "${GROOT_ROOT}"
source .venv/bin/activate
nohup uv run python gr00t/eval/run_gr00t_server.py \
  --model-path "${MODEL_PATH}" \
  --embodiment-tag "${EMBODIMENT_TAG}" \
  --use-sim-policy-wrapper >"${UPSTREAM_LOG}" 2>&1 &

cd /content/robotic-task-breaker
uv pip install --python "${GROOT_PY}" fastapi "uvicorn[standard]" python-multipart pillow
for _ in $(seq 1 180); do
  if curl -fsS http://127.0.0.1:5555 >/dev/null 2>&1 || grep -q 'listening' "${UPSTREAM_LOG}" 2>/dev/null; then
    break
  fi
  sleep 5
done

nohup "${GROOT_PY}" services/run_vla_server.py >"${ADAPTER_LOG}" 2>&1 &

for _ in $(seq 1 120); do
  HEALTH_JSON="$(curl -fsS http://127.0.0.1:8765/health 2>/dev/null || true)"
  if [ -n "${HEALTH_JSON}" ]; then
    echo "${HEALTH_JSON}"
    echo
    echo 'Robotic Task Breaker GR00T adapter healthy'
    exit 0
  fi
  sleep 5
done

echo 'adapter failed to become healthy' >&2
tail -50 "${ADAPTER_LOG}" >&2
exit 1

## 6 — Run Robotic Task Breaker `X` episodes


In [ ]:
%%bash
set -euo pipefail
source /content/robotic-task-breaker/.colab_m2_env
cd /content/robotic-task-breaker

ROBOTIC_TASK_BREAKER_DIR="${ARTIFACT_DIR}/robotic_task_breaker"
mkdir -p "${ROBOTIC_TASK_BREAKER_DIR}"
"${ROBO_PY}" validation/run_robotic_task_breaker_arm.py \
  --episodes "${M4_X}" \
  --artifact-dir "${ROBOTIC_TASK_BREAKER_DIR}"


## 7 — Run bare policy `X` episodes


In [ ]:
%%bash
set -euo pipefail
source /content/robotic-task-breaker/.colab_m2_env
cd /content/robotic-task-breaker

BARE_DIR="${ARTIFACT_DIR}/bare"
mkdir -p "${BARE_DIR}"
"${ROBO_PY}" validation/run_bare_policy.py \
  --episodes "${M4_X}" \
  --artifact-dir "${BARE_DIR}"


## 8 — Interim stats (20/arm early stop)


In [ ]:
import json
import os
import subprocess
from pathlib import Path

artifact_dir = Path(os.environ['ARTIFACT_DIR'])
robotic_task_breaker_dir = artifact_dir / 'robotic_task_breaker'
bare_dir = artifact_dir / 'bare'
verdict_path = artifact_dir / 'm4_interim_verdict.json'

proc = subprocess.run(
    [
        'python',
        'validation/m4_unpaired_stats.py',
        '--mode',
        'interim',
        '--robotic-task-breaker-summary',
        str(robotic_task_breaker_dir / 'robotic_task_breaker_summary.json'),
        '--bare-summary',
        str(bare_dir / 'bare_summary.json'),
        '--output',
        str(verdict_path),
    ],
    cwd='/content/robotic-task-breaker',
    check=True,
    text=True,
    capture_output=True,
)
print(proc.stdout)
interim = json.loads(verdict_path.read_text(encoding='utf-8'))
os.environ['M4_INTERIM_CLAIM'] = '1' if interim.get('claim') else '0'
print('M4_INTERIM_CLAIM=', os.environ['M4_INTERIM_CLAIM'])


## 9 — Continue to 50/arm and final stats if interim is not decisive


In [ ]:
%%bash
set -euo pipefail
source /content/robotic-task-breaker/.colab_m2_env
cd /content/robotic-task-breaker

INTERIM_CLAIM=$("${ROBO_PY}" - <<'PY'
import json, os
from pathlib import Path
p = Path(os.environ['ARTIFACT_DIR']) / 'm4_interim_verdict.json'
print('1' if json.loads(p.read_text(encoding='utf-8')).get('claim') else '0')
PY
)
if [ "${INTERIM_CLAIM}" = "1" ]; then
  echo 'Interim claim met; skipping final batch.'
  exit 0
fi

EXTRA=$((M4_FINAL_N - M4_X))
if [ "${EXTRA}" -le 0 ]; then
  echo 'Nothing to add for final batch.'
  exit 0
fi

ROBOTIC_TASK_BREAKER_DIR="${ARTIFACT_DIR}/robotic_task_breaker"
BARE_DIR="${ARTIFACT_DIR}/bare"
"${ROBO_PY}" validation/run_robotic_task_breaker_arm.py \
  --episodes "${EXTRA}" \
  --start-index "${M4_X}" \
  --append \
  --artifact-dir "${ROBOTIC_TASK_BREAKER_DIR}"
"${ROBO_PY}" validation/run_bare_policy.py \
  --episodes "${EXTRA}" \
  --start-index "${M4_X}" \
  --append \
  --artifact-dir "${BARE_DIR}"

"${ROBO_PY}" validation/m4_unpaired_stats.py \
  --mode final \
  --robotic-task-breaker-summary "${ROBOTIC_TASK_BREAKER_DIR}/robotic_task_breaker_summary.json" \
  --bare-summary "${BARE_DIR}/bare_summary.json" \
  --output "${ARTIFACT_DIR}/m4_final_verdict.json"
